# Kazakh News Classification: Knowledge Distillation & Quantization

This notebook implements the full pipeline described in the paper:
> **"Evaluating the Impact of Knowledge Distillation and Weight Quantization on Transformer-Based Model Optimization for the Kazakh Language"**

**Pipeline stages:**
1. Dataset loading & preprocessing
2. Teacher model training (XLM-RoBERTa Large)
3. Knowledge distillation → KazRoBERTa student
4. Quantization evaluation (FP16 / INT8 / NF4)
5. Inference examples

**Requirements:** GPU with ≥ 16 GB VRAM (A100 40 GB recommended for full pipeline)


## 0. Install dependencies

In [ ]:
# Install required packages
!pip install -q transformers datasets accelerate evaluate scikit-learn bitsandbytes tensorboard


## 1. Imports & reproducibility

In [ ]:
import os, re, json
import torch
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import f1_score, classification_report
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoConfig,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    BitsAndBytesConfig,
)

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    print(f"GPU : {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("No GPU found — some cells will be slow or unavailable")


## 2. Dataset loading & preprocessing

The dataset is from the Kazakh newspaper *Egemen Qazaqstan*.
It is publicly available on Hugging Face: `AishaSailau/news_kk_v2`.

You can also use the CSV snapshot from the paper's GitHub repository:
`https://github.com/nazerkegalymkyzy97/data/blob/main/Desktop/dataset-kaz/df20.csv`


In [ ]:
# ── Option A: load from Hugging Face Datasets ──────────────────────────────
# Requires a Hugging Face account token if the dataset is private.
# Set the HF_TOKEN environment variable before running:
#   export HF_TOKEN=<your_token>   (Linux/Mac)
#   $env:HF_TOKEN="<your_token>"  (Windows PowerShell)
#
# from huggingface_hub import login
# login(token=os.environ.get("HF_TOKEN"))  # reads token from env, never hard-code it

# Uncomment to load directly:
# df_raw = pd.read_json("hf://datasets/AishaSailau/news_kk_v2/news_kk_filtered_removed.json")

# ── Option B: load from the CSV snapshot ──────────────────────────────────
# df_raw = pd.read_csv("df20.csv")

# ── For this notebook we use Option A (comment/uncomment as needed) ────────
from datasets import load_dataset
ds  = load_dataset("AishaSailau/news_kk_v2")
df_raw = ds["train"].to_pandas()

print(f"Raw dataset shape: {df_raw.shape}")
print(df_raw.head(2))


### 2.1 Category selection & text cleaning

In [ ]:
# Categories with ≥ 500 samples in the original corpus
KEEP_CATEGORIES = [
    'Аймақтар', 'Қоғам', 'Әлем', 'Спорт', 'Президент', 'Оқиға',
    'Білім', 'Елорда', 'Экономика', 'Саясат', 'Медицина', 'Өнер',
    'Ауа райы', 'Парламент',
]

df = df_raw[df_raw['category'].isin(KEEP_CATEGORIES)][['abstract', 'category']].copy()
df = df.dropna(subset=['abstract', 'category'])
df = df.drop_duplicates(subset=['abstract'])
df['abstract'] = df['abstract'].str.strip()

# Length-based filtering (≥ 5 words, as per paper §2.1)
def clean_text(text: str) -> str:
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'[^\w\s.,!?;:%-]', '', text)
    return text.strip()

df['abstract'] = df['abstract'].apply(clean_text)
df['length']   = df['abstract'].apply(lambda x: len(x.split()))
df = df[df['length'] >= 5].reset_index(drop=True)

print(f"After filtering: {df.shape}")
print(df['category'].value_counts())


### 2.2 Train / val / test split + class balancing

In [ ]:
# ── Label encoding ──────────────────────────────────────────────────────────
label2id = {c: i for i, c in enumerate(sorted(df['category'].unique()))}
id2label = {v: k for k, v in label2id.items()}
df['label'] = df['category'].map(label2id)
NUM_LABELS = len(label2id)
print(f"Number of classes: {NUM_LABELS}")

# ── Stratified split 80 / 10 / 10 ─────────────────────────────────────────
train_df, temp_df = train_test_split(df, test_size=0.2, stratify=df['label'], random_state=SEED)
val_df,   test_df = train_test_split(temp_df, test_size=0.5, stratify=temp_df['label'], random_state=SEED)

# ── Distribution-controlled balancing (§2.1) ───────────────────────────────
MIN_COUNT, MAX_COUNT = 1000, 2000

def balance_dataset(df, min_count=MIN_COUNT, max_count=MAX_COUNT, seed=SEED):
    parts = []
    for _, grp in df.groupby('label'):
        n = len(grp)
        if n < min_count:
            print(f"  REMOVED {grp['category'].iloc[0]}: {n} samples")
            continue
        parts.append(grp.sample(min(n, max_count), replace=False, random_state=seed))
    return pd.concat(parts).sample(frac=1, random_state=seed).reset_index(drop=True)

print("Balancing train set...")
train_balanced = balance_dataset(train_df)

# Re-align val/test to surviving categories
remaining  = sorted(train_balanced['category'].unique())
label2id   = {c: i for i, c in enumerate(remaining)}
id2label   = {v: k for k, v in label2id.items()}
NUM_LABELS = len(label2id)

train_balanced['label'] = train_balanced['category'].map(label2id)
val_df   = val_df[val_df['category'].isin(remaining)].copy()
test_df  = test_df[test_df['category'].isin(remaining)].copy()
val_df['label']  = val_df['category'].map(label2id)
test_df['label'] = test_df['category'].map(label2id)

print(f"\nFinal split — train: {len(train_balanced)} | val: {len(val_df)} | test: {len(test_df)}")
print("Train distribution:")
print(train_balanced['category'].value_counts())


## 3. Teacher model training (XLM-RoBERTa Large)

The teacher is XLM-RoBERTa Large (560M parameters).
It is trained with focal loss + class weighting to handle residual imbalance.

> ⏱ ~1–2 h on A100 40 GB


In [ ]:
TEACHER_CFG = {
    "model_name": "xlm-roberta-large",
    "max_len":    128,
    "batch_size": 32,
    "grad_accum": 4,
    "lr":         5e-5,
    "max_steps":  3000,
    "warmup_steps": 300,
    "output_dir": "./teacher_xlmr_large",
}


In [ ]:
class_weights = compute_class_weight(
    'balanced',
    classes=np.arange(NUM_LABELS),
    y=train_balanced['label'].values
)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32)
print(f"Class weights  min={class_weights.min():.3f}  max={class_weights.max():.3f}")


In [ ]:
tokenizer_t = AutoTokenizer.from_pretrained(TEACHER_CFG["model_name"])

class NewsDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=128):
        self.texts  = df['abstract'].fillna('').tolist()
        self.labels = df['label'].tolist()
        self.tok    = tokenizer
        self.maxlen = max_len

    def __len__(self): return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tok(self.texts[idx], max_length=self.maxlen,
                       padding='max_length', truncation=True, return_tensors='pt')
        return {
            'input_ids':      enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'labels':         torch.tensor(self.labels[idx], dtype=torch.long),
        }

train_ds_t = NewsDataset(train_balanced, tokenizer_t, TEACHER_CFG['max_len'])
val_ds_t   = NewsDataset(val_df,         tokenizer_t, TEACHER_CFG['max_len'])
test_ds_t  = NewsDataset(test_df,        tokenizer_t, TEACHER_CFG['max_len'])

model_t = AutoModelForSequenceClassification.from_pretrained(
    TEACHER_CFG['model_name'], num_labels=NUM_LABELS,
    id2label=id2label, label2id=label2id,
)


In [ ]:
class FocalLoss(torch.nn.Module):
    def __init__(self, alpha=None, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits, labels):
        ce  = F.cross_entropy(logits, labels,
                              weight=self.alpha.to(logits.device) if self.alpha is not None else None,
                              reduction='none')
        pt   = torch.exp(-ce)
        loss = ((1 - pt) ** self.gamma) * ce
        return loss.mean()

class WeightedTrainer(Trainer):
    def __init__(self, class_weights, **kwargs):
        super().__init__(**kwargs)
        self.focal = FocalLoss(alpha=class_weights, gamma=2.0)

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels  = inputs.pop('labels')
        outputs = model(**inputs)
        loss    = self.focal(outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'macro_f1': f1_score(labels, preds, average='macro'),
        'accuracy': (preds == labels).mean(),
    }

teacher_args = TrainingArguments(
    output_dir=TEACHER_CFG['output_dir'],
    max_steps=TEACHER_CFG['max_steps'],
    per_device_train_batch_size=TEACHER_CFG['batch_size'],
    per_device_eval_batch_size=64,
    gradient_accumulation_steps=TEACHER_CFG['grad_accum'],
    fp16=True,
    learning_rate=TEACHER_CFG['lr'],
    lr_scheduler_type='cosine',
    warmup_steps=TEACHER_CFG['warmup_steps'],
    weight_decay=0.01,
    label_smoothing_factor=0.1,
    eval_strategy='steps', eval_steps=300,
    save_strategy='steps', save_steps=300,
    load_best_model_at_end=True,
    metric_for_best_model='macro_f1',
    greater_is_better=True,
    save_total_limit=2,
    logging_steps=50,
    report_to='none',
    seed=SEED,
)

teacher_trainer = WeightedTrainer(
    class_weights=class_weights_tensor,
    model=model_t, args=teacher_args,
    train_dataset=train_ds_t, eval_dataset=val_ds_t,
    processing_class=tokenizer_t,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=5)],
)

print("Starting teacher training ...")
teacher_trainer.train()
teacher_trainer.save_model(f"{TEACHER_CFG['output_dir']}/best")
tokenizer_t.save_pretrained(f"{TEACHER_CFG['output_dir']}/best")
print("Teacher saved!")


In [ ]:
# Evaluate teacher on test set
preds_t = teacher_trainer.predict(test_ds_t)
y_pred_t = np.argmax(preds_t.predictions, axis=-1)
y_true_t = preds_t.label_ids
print(f"Teacher  Macro F1 : {f1_score(y_true_t, y_pred_t, average='macro'):.4f}")
print(f"Teacher  Accuracy : {(y_pred_t == y_true_t).mean():.4f}")
print(classification_report(y_true_t, y_pred_t,
      target_names=[id2label[i] for i in range(NUM_LABELS)], digits=4))


## 4. Knowledge Distillation → KazRoBERTa Student

Soft-target KD (Hinton et al., 2015).
The student is `kz-transformers/kaz-roberta-conversational` (83M parameters), giving a **6.7× compression** vs. the teacher.

Loss: `L = α · L_KD  +  (1-α) · L_CE`

> ⏱ ~30–40 min on A100


In [ ]:
DISTILL_CFG = {
    "teacher_path": "./teacher_xlmr_large/best",
    "student_name": "kz-transformers/kaz-roberta-conversational",
    "temperature":  4.0,    # T > 1 softens teacher distribution
    "alpha":        0.7,    # weight of KD loss
    "max_len":      128,
    "batch_size":   64,
    "grad_accum":   2,
    "lr":           3e-5,
    "max_steps":    2000,
    "warmup_steps": 200,
    "output_dir":   "./student_kazroberta",
}


In [ ]:
# ── Load teacher (frozen) ─────────────────────────────────────────────────
print("Loading teacher ...")
teacher_tok_d  = AutoTokenizer.from_pretrained(DISTILL_CFG['teacher_path'])
teacher_mod_d  = AutoModelForSequenceClassification.from_pretrained(
    DISTILL_CFG['teacher_path']).to(device)
teacher_mod_d.eval()
for p in teacher_mod_d.parameters():
    p.requires_grad = False
teacher_params = sum(p.numel() for p in teacher_mod_d.parameters()) / 1e6
print(f"Teacher: {teacher_params:.1f}M params")

# ── Load student ──────────────────────────────────────────────────────────
print(f"Loading student: {DISTILL_CFG['student_name']} ...")
student_tok  = AutoTokenizer.from_pretrained(DISTILL_CFG['student_name'])
student_mod  = AutoModelForSequenceClassification.from_pretrained(
    DISTILL_CFG['student_name'],
    num_labels=NUM_LABELS, id2label=id2label, label2id=label2id,
    ignore_mismatched_sizes=True,
)
student_params = sum(p.numel() for p in student_mod.parameters()) / 1e6
print(f"Student: {student_params:.1f}M params  |  compression {teacher_params/student_params:.1f}x")


In [ ]:
# ── Precompute teacher soft targets (single forward pass) ─────────────────
def precompute_soft_targets(texts, tokenizer, model, temperature, max_len=128, batch_size=128):
    all_probs, model.eval = [], model.eval
    model.eval()
    for i in range(0, len(texts), batch_size):
        batch = texts[i: i+batch_size]
        enc   = tokenizer(batch, max_length=max_len, padding='max_length',
                          truncation=True, return_tensors='pt')
        enc   = {k: v.to(device) for k, v in enc.items()}
        with torch.no_grad():
            logits = model(**enc).logits
            probs  = F.softmax(logits / temperature, dim=-1)
            all_probs.append(probs.cpu())
        print(f"  {min(i+batch_size, len(texts))}/{len(texts)}", end="\r")
    print()
    return torch.cat(all_probs, dim=0)

print("Precomputing soft targets ...")
T = DISTILL_CFG['temperature']
train_soft = precompute_soft_targets(train_balanced['abstract'].fillna('').tolist(), teacher_tok_d, teacher_mod_d, T)
val_soft   = precompute_soft_targets(val_df['abstract'].fillna('').tolist(),         teacher_tok_d, teacher_mod_d, T)
test_soft  = precompute_soft_targets(test_df['abstract'].fillna('').tolist(),        teacher_tok_d, teacher_mod_d, T)
print(f"Shapes — train {train_soft.shape} | val {val_soft.shape} | test {test_soft.shape}")


In [ ]:
class DistillDataset(Dataset):
    def __init__(self, df, tokenizer, soft_targets, max_len=128):
        self.texts        = df['abstract'].fillna('').tolist()
        self.labels       = df['label'].tolist()
        self.soft_targets = soft_targets
        self.tok          = tokenizer
        self.maxlen       = max_len

    def __len__(self): return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tok(self.texts[idx], max_length=self.maxlen,
                       padding='max_length', truncation=True, return_tensors='pt')
        return {
            'input_ids':      enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'soft_targets':   self.soft_targets[idx],
            'labels':         torch.tensor(self.labels[idx], dtype=torch.long),
        }

class DistillationTrainer(Trainer):
    def __init__(self, temperature, alpha, **kwargs):
        super().__init__(**kwargs)
        self.temperature = temperature
        self.alpha       = alpha

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        soft_targets = inputs.pop('soft_targets')
        labels       = inputs.pop('labels')
        outputs      = model(**inputs)
        logits       = outputs.logits
        T            = self.temperature

        kd_loss = F.kl_div(F.log_softmax(logits / T, dim=-1),
                           soft_targets.to(logits.device),
                           reduction='batchmean') * (T ** 2)
        ce_loss = F.cross_entropy(logits, labels)
        loss    = self.alpha * kd_loss + (1 - self.alpha) * ce_loss
        return (loss, outputs) if return_outputs else loss

train_dd = DistillDataset(train_balanced, student_tok, train_soft, DISTILL_CFG['max_len'])
val_dd   = DistillDataset(val_df,         student_tok, val_soft,   DISTILL_CFG['max_len'])
test_dd  = DistillDataset(test_df,        student_tok, test_soft,  DISTILL_CFG['max_len'])

distill_args = TrainingArguments(
    output_dir=DISTILL_CFG['output_dir'],
    max_steps=DISTILL_CFG['max_steps'],
    per_device_train_batch_size=DISTILL_CFG['batch_size'],
    per_device_eval_batch_size=64,
    gradient_accumulation_steps=DISTILL_CFG['grad_accum'],
    fp16=True,
    learning_rate=DISTILL_CFG['lr'],
    lr_scheduler_type='cosine',
    warmup_steps=DISTILL_CFG['warmup_steps'],
    weight_decay=0.01,
    eval_strategy='steps', eval_steps=200,
    save_strategy='steps', save_steps=200,
    load_best_model_at_end=True,
    metric_for_best_model='macro_f1',
    greater_is_better=True,
    save_total_limit=2,
    logging_steps=50,
    report_to='none',
    remove_unused_columns=False,
    seed=SEED,
)

distill_trainer = DistillationTrainer(
    temperature=DISTILL_CFG['temperature'],
    alpha=DISTILL_CFG['alpha'],
    model=student_mod, args=distill_args,
    train_dataset=train_dd, eval_dataset=val_dd,
    processing_class=student_tok,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=5)],
)

print("Starting distillation ...")
distill_trainer.train()
distill_trainer.save_model(f"{DISTILL_CFG['output_dir']}/best")
student_tok.save_pretrained(f"{DISTILL_CFG['output_dir']}/best")
print("Student saved!")


In [ ]:
preds_s  = distill_trainer.predict(test_dd)
y_pred_s = np.argmax(preds_s.predictions, axis=-1)
y_true_s = preds_s.label_ids
print(f"Student  Macro F1 : {f1_score(y_true_s, y_pred_s, average='macro'):.4f}")
print(f"Student  Accuracy : {(y_pred_s == y_true_s).mean():.4f}")
print(classification_report(y_true_s, y_pred_s,
      target_names=[id2label[i] for i in range(NUM_LABELS)], digits=4))


## 5. Quantization evaluation

We benchmark the distilled student model at three precision levels:
| Method | Bits | Library |
|--------|------|---------|
| FP16   | 16   | PyTorch native |
| INT8   | 8    | BitsAndBytes `LLM.int8()` |
| NF4    | 4    | BitsAndBytes NormalFloat-4 |

The classifier head is excluded from quantization in all cases.


In [ ]:
Q_CFG = {
    "model_path": "./student_kazroberta/best",   # path to distilled student
}

# ── Benchmark helper ──────────────────────────────────────────────────────
def benchmark(model, df, label_col='label', max_len=128, batch_size=64, label='model'):
    tok = AutoTokenizer.from_pretrained(Q_CFG['model_path'])
    ds  = NewsDataset(df, tok, max_len)
    loader = DataLoader(ds, batch_size=batch_size, pin_memory=True)

    model.eval()
    all_preds, all_labels, times = [], [], []
    import time

    with torch.no_grad():
        for batch in loader:
            ids  = batch['input_ids'].to(device)
            mask = batch['attention_mask'].to(device)
            lbls = batch['labels'].cpu().numpy()

            t0     = time.perf_counter()
            logits = model(ids, attention_mask=mask).logits
            times.append((time.perf_counter() - t0) * 1000)

            all_preds.extend(logits.argmax(-1).cpu().numpy())
            all_labels.extend(lbls)

    f1  = f1_score(all_labels, all_preds, average='macro')
    acc = (np.array(all_preds) == np.array(all_labels)).mean()
    lat = np.mean(times)

    # Memory
    vram = torch.cuda.memory_allocated() / 1e6
    size = sum(p.numel() * p.element_size() for p in model.parameters()) / 1e6

    print(f"{label:<35}  F1={f1:.4f}  Acc={acc:.4f}  Lat={lat:.1f}ms  Size={size:.0f}MB  VRAM={vram:.0f}MB")
    return dict(label=label, f1=f1, acc=acc, lat_ms=lat, size_mb=size, vram_mb=vram,
                preds=all_preds, labels=all_labels)

results = {}


In [ ]:
# ── FP16 ──────────────────────────────────────────────────────────────────
print("Loading FP16 ...")
m_fp16 = AutoModelForSequenceClassification.from_pretrained(
    Q_CFG['model_path'], torch_dtype=torch.float16).to(device)
results['fp16'] = benchmark(m_fp16, test_df, label='KazRoBERTa FP16 (16-bit)')
del m_fp16; torch.cuda.empty_cache()


In [ ]:
# ── INT8 ──────────────────────────────────────────────────────────────────
print("Loading INT8 ...")
m_int8 = AutoModelForSequenceClassification.from_pretrained(
    Q_CFG['model_path'],
    quantization_config=BitsAndBytesConfig(
        load_in_8bit=True,
        llm_int8_threshold=6.0,
        llm_int8_skip_modules=['classifier', 'pooler'],
    ),
    device_map='auto',
)
results['int8'] = benchmark(m_int8, test_df, label='BitsAndBytes INT8 (8-bit)')
del m_int8; torch.cuda.empty_cache()


In [ ]:
# ── NF4 (4-bit NormalFloat) ───────────────────────────────────────────────
print("Loading NF4 ...")
m_nf4 = AutoModelForSequenceClassification.from_pretrained(
    Q_CFG['model_path'],
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
        llm_int8_skip_modules=['classifier', 'pooler'],
    ),
    device_map='auto',
)
results['nf4'] = benchmark(m_nf4, test_df, label='BitsAndBytes NF4  (4-bit)')
del m_nf4; torch.cuda.empty_cache()


In [ ]:
# ── Summary table ─────────────────────────────────────────────────────────
ref = results['fp16']
print("\n" + "="*80)
print(f"{'Method':<35} {'F1':>7} {'ΔF1':>7} {'Lat ms':>8} {'Speedup':>9} {'Size MB':>9}")
print("─"*80)
for k, r in results.items():
    df_val = r['f1'] - ref['f1']
    sp     = ref['lat_ms'] / r['lat_ms']
    print(f"{r['label']:<35} {r['f1']:>7.4f} {df_val:>+7.4f} {r['lat_ms']:>8.1f} {sp:>8.2f}x {r['size_mb']:>9.0f}")
print("="*80)


## 6. Inference examples

Run the distilled model on your own Kazakh text.


In [ ]:
def load_model_for_inference(model_path: str, precision: str = 'fp16'):
    """
    precision: 'fp16' | 'int8' | 'nf4'
    """
    tok = AutoTokenizer.from_pretrained(model_path)

    if precision == 'int8':
        qcfg = BitsAndBytesConfig(load_in_8bit=True, llm_int8_skip_modules=['classifier', 'pooler'])
        model = AutoModelForSequenceClassification.from_pretrained(
            model_path, quantization_config=qcfg, device_map='auto')
    elif precision == 'nf4':
        qcfg = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                                  bnb_4bit_compute_dtype=torch.float16,
                                  bnb_4bit_use_double_quant=True,
                                  llm_int8_skip_modules=['classifier', 'pooler'])
        model = AutoModelForSequenceClassification.from_pretrained(
            model_path, quantization_config=qcfg, device_map='auto')
    else:
        model = AutoModelForSequenceClassification.from_pretrained(
            model_path, torch_dtype=torch.float16).to(device)

    cfg = AutoConfig.from_pretrained(model_path)
    _id2label = {int(k): v for k, v in cfg.id2label.items()}
    return model, tok, _id2label


def predict(text: str, model, tokenizer, id2label, max_len=128, top_k=3):
    model.eval()
    enc = tokenizer(text, max_length=max_len, padding='max_length',
                    truncation=True, return_tensors='pt')
    enc = {k: v.to(device) for k, v in enc.items()}
    with torch.no_grad():
        logits = model(**enc).logits.squeeze(0)
    probs   = torch.softmax(logits, dim=-1).cpu().numpy()
    topk    = probs.argsort()[::-1][:top_k]
    return id2label[int(topk[0])], float(probs[topk[0]]), [(id2label[i], float(probs[i])) for i in topk]


In [ ]:
# Load distilled model (change precision to 'int8' or 'nf4' as needed)
inf_model, inf_tok, inf_id2label = load_model_for_inference(
    Q_CFG['model_path'], precision='fp16'
)

# ── Example texts ─────────────────────────────────────────────────────────
examples = [
    "Ташкент қаласында өткен көркем гимнастикадан Әлем кубогінің үшінші кезеңінде "
    "астаналық «Фурор» командасы жүлдегер атанды.",
    "Қазақстан Үкіметі шағын және орта бизнесті қолдау үшін жаңа салықтық "
    "жеңілдіктер пакетін бекітті.",
    "Л.Н. Гумилев атындағы Еуразия ұлттық университетінде Жасанды интеллект "
    "және этика атты воркшоп өтті.",
]

print("="*60)
for i, text in enumerate(examples, 1):
    pred, conf, topk = predict(text, inf_model, inf_tok, inf_id2label)
    print(f"\nExample {i}: {text[:80]}...")
    print(f"  Predicted: {pred}  ({conf*100:.1f}%)")
    for rank, (lbl, prob) in enumerate(topk, 1):
        print(f"    {rank}. {lbl:<15} {prob*100:5.1f}%  {'█'*int(prob*30)}")
print("="*60)


## 7. Quick start: use the pretrained model from Hugging Face

If you just want to run inference without training, use the fine-tuned model
published alongside this paper.

```
https://huggingface.co/Rniwd/Kaz_Roberta_fine_tuned
```


In [ ]:
# Load the published fine-tuned model directly (no training required)
PUBLISHED_MODEL = "Rniwd/Kaz_Roberta_fine_tuned"

pub_model, pub_tok, pub_id2label = load_model_for_inference(PUBLISHED_MODEL, precision='fp16')

text = "Президент Тоқаев Ұлттық банк төрағасымен экономикалық жағдайды талқылады."
pred, conf, topk = predict(text, pub_model, pub_tok, pub_id2label)
print(f"Text     : {text}")
print(f"Predicted: {pred}  ({conf*100:.1f}%)")
for r, (lbl, prob) in enumerate(topk, 1):
    print(f"  {r}. {lbl:<16} {prob*100:5.1f}%")
